# Testing RSHAP

### Unit Test Suite

In [ ]:
sys.path.insert(0, r"C:\Users\Amanda\Favorites\Machine_Learning\SOFTWARE\RSHAP")

In [3]:

import sys, warnings
sys.path.insert(0, "/path/to/RSHAP")
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")

import unittest
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from xgboost import XGBRegressor, XGBClassifier

from RSHAP import (
    ResidualDecompositionSymmetric,
    resolve_model,
    _compute_marginal_contributions,
    draw_heatmap,
    _MODEL_REGISTRY,
)

# ── Shared tiny datasets ──────────────────────────────────────────────────────
np.random.seed(0)
N, F = 20, 3
Xr = np.random.randn(N, F)
yr = (Xr @ [1.5, -1.0, 0.5] + np.random.randn(N) * 0.2).astype(np.float64)
Xc = np.random.randn(N, F)
yc = (Xc[:, 0] > 0).astype(int)


# ── 1. resolve_model ─────────────────────────────────────────────────────────
class TestResolveModel(unittest.TestCase):

    def test_registry_has_expected_keys(self):
        for key in ('ridge', 'logistic', 'svm', 'rf', 'mlp', 'xgb'):
            self.assertIn(key, _MODEL_REGISTRY)

    def test_ridge_regression(self):
        cls, params = resolve_model('ridge', None, True)
        self.assertIs(cls, Ridge);  self.assertEqual(params, {})

    def test_ridge_returns_logistic_for_classification(self):
        cls, _ = resolve_model('ridge', None, False)
        self.assertIs(cls, LogisticRegression)

    def test_logistic_alias(self):
        cls, _ = resolve_model('logistic', None, False)
        self.assertIs(cls, LogisticRegression)

    def test_svm_regression(self):
        self.assertIs(resolve_model('svm', None, True)[0], SVR)

    def test_svm_classification(self):
        self.assertIs(resolve_model('svm', None, False)[0], SVC)

    def test_rf_regression(self):
        self.assertIs(resolve_model('rf', None, True)[0], RandomForestRegressor)

    def test_rf_classification(self):
        self.assertIs(resolve_model('rf', None, False)[0], RandomForestClassifier)

    def test_mlp_regression(self):
        self.assertIs(resolve_model('mlp', None, True)[0], MLPRegressor)

    def test_mlp_classification(self):
        self.assertIs(resolve_model('mlp', None, False)[0], MLPClassifier)

    def test_xgb_regression(self):
        self.assertIs(resolve_model('xgb', None, True)[0], XGBRegressor)

    def test_xgb_classification(self):
        self.assertIs(resolve_model('xgb', None, False)[0], XGBClassifier)

    def test_case_insensitive(self):
        self.assertIs(resolve_model('RF', None, True)[0], RandomForestRegressor)
        self.assertIs(resolve_model('SVM', None, False)[0], SVC)
        self.assertIs(resolve_model('XGB', None, True)[0], XGBRegressor)

    def test_passthrough_class_object(self):
        cls, params = resolve_model(Ridge, {'alpha': 2.0}, True)
        self.assertIs(cls, Ridge);  self.assertEqual(params, {'alpha': 2.0})

    def test_none_params_defaults_to_empty_dict(self):
        _, params = resolve_model(Ridge, None, True)
        self.assertEqual(params, {})

    def test_explicit_params_preserved(self):
        _, params = resolve_model('svm', {'C': 5.0, 'kernel': 'rbf'}, True)
        self.assertEqual(params, {'C': 5.0, 'kernel': 'rbf'})

    def test_invalid_name_raises_value_error(self):
        with self.assertRaises(ValueError):
            resolve_model('neural_net', None, True)

    def test_error_message_lists_valid_names(self):
        with self.assertRaises(ValueError) as ctx:
            resolve_model('bad_model', None, True)
        self.assertIn('ridge', str(ctx.exception).lower())


# ── 2. _compute_marginal_contributions ───────────────────────────────────────
class TestComputeMarginalContributions(unittest.TestCase):

    def setUp(self):
        np.random.seed(7)
        self.perm = np.random.permutation(N)

    def test_returns_ndarray(self):
        result = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        self.assertIsInstance(result, np.ndarray)

    def test_shape_regression(self):
        result = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        self.assertEqual(result.shape, (N, N))

    def test_dtype_float64(self):
        result = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        self.assertEqual(result.dtype, np.float64)

    def test_no_nan_regression(self):
        result = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        self.assertFalse(np.any(np.isnan(result)))

    def test_shape_classification(self):
        result = _compute_marginal_contributions(Xc, yc, LogisticRegression, {}, self.perm, False)
        self.assertEqual(result.shape, (N, N))

    def test_no_nan_classification(self):
        result = _compute_marginal_contributions(Xc, yc, LogisticRegression, {}, self.perm, False)
        self.assertFalse(np.any(np.isnan(result)))

    def test_classification_values_are_integers(self):
        # classification residuals are binarised to 0/1, so phi entries are multiples of 1
        result = _compute_marginal_contributions(Xc, yc, LogisticRegression, {}, self.perm, False)
        self.assertTrue(np.all(result == result.astype(int)))

    def test_symmetric_pair_different(self):
        # forward and reverse permutations should not be identical
        fwd = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        rev = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm[::-1], True)
        self.assertFalse(np.allclose(fwd, rev))

    def test_returns_none_on_model_crash(self):
        class CrashModel:
            def fit(self, X, y): raise RuntimeError("deliberate crash")
        result = _compute_marginal_contributions(Xr, yr, CrashModel, {}, self.perm, True)
        self.assertIsNone(result)

    def test_deterministic_with_fixed_permutation(self):
        r1 = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        r2 = _compute_marginal_contributions(Xr, yr, Ridge, {}, self.perm, True)
        np.testing.assert_array_equal(r1, r2)


# ── 3. ResidualDecompositionSymmetric ────────────────────────────────────────
def _fit(X, y, model='ridge', regression=True, iters=4):
    r = ResidualDecompositionSymmetric()
    r.fit(X, y, model_class=model, iterations=iters, regression=regression, n_jobs=1)
    return r

class TestRSHAPAttributes(unittest.TestCase):

    def test_phi_exists_after_fit(self):
        r = _fit(Xr, yr)
        self.assertTrue(hasattr(r, 'phi'))

    def test_composition_exists_after_fit(self):
        r = _fit(Xr, yr)
        self.assertTrue(hasattr(r, 'composition'))

    def test_n_jobs_stored(self):
        r = ResidualDecompositionSymmetric()
        r.fit(Xr, yr, model_class='ridge', iterations=4, n_jobs=3)
        self.assertEqual(r.n_jobs, 3)

    def test_model_class_resolved(self):
        r = _fit(Xr, yr, model='ridge')
        self.assertIs(r.model_class, Ridge)

    def test_model_params_default_empty(self):
        r = _fit(Xr, yr)
        self.assertEqual(r.model_params, {})

    def test_regression_flag_stored(self):
        r = _fit(Xc, yc, regression=False)
        self.assertFalse(r.r)


class TestRSHAPOutputs(unittest.TestCase):

    def setUp(self):
        self.r_reg = _fit(Xr, yr, regression=True)
        self.r_clf = _fit(Xc, yc, regression=False)

    def test_composition_shape_regression(self):
        self.assertEqual(self.r_reg.get_composition().shape, (N, N))

    def test_contribution_shape_regression(self):
        self.assertEqual(self.r_reg.get_contribution().shape, (N, N))

    def test_composition_shape_classification(self):
        self.assertEqual(self.r_clf.get_composition().shape, (N, N))

    def test_phi_not_all_zeros_regression(self):
        self.assertFalse(np.all(self.r_reg.phi == 0))

    def test_phi_not_all_zeros_classification(self):
        self.assertFalse(np.all(self.r_clf.phi == 0))

    def test_no_nan_composition(self):
        self.assertFalse(np.any(np.isnan(self.r_reg.get_composition())))

    def test_no_nan_contribution(self):
        self.assertFalse(np.any(np.isnan(self.r_reg.get_contribution())))

    def test_composition_equals_phi(self):
        np.testing.assert_array_equal(
            self.r_reg.get_composition(), self.r_reg.phi)

    def test_contribution_rows_scaled_by_neg_sign_of_col_sums(self):
        # get_contribution() transposes, multiplies rows by -sign(col_sum), then transposes back.
        # Net effect: contrib[i, j] = comp[i, j] * -sign(sum_k comp[k, i])
        # i.e. every element in row i is scaled by -sign(col_sum of column i).
        comp     = self.r_reg.get_composition()
        contrib  = self.r_reg.get_contribution()
        col_sums = np.sum(comp, axis=0)                        # shape (N,)
        expected = comp * (-np.sign(col_sums))[:, np.newaxis]  # broadcast over columns
        np.testing.assert_allclose(contrib, expected, atol=1e-10)

    def test_class_object_accepted(self):
        r = ResidualDecompositionSymmetric()
        r.fit(Xr, yr, model_class=Ridge, model_params={'alpha': 0.5}, iterations=4, n_jobs=1)
        self.assertEqual(r.get_composition().shape, (N, N))

    def test_more_iterations_changes_phi(self):
        r4  = _fit(Xr, yr, iters=4)
        r20 = _fit(Xr, yr, iters=20)
        self.assertFalse(np.allclose(r4.phi, r20.phi))

    def test_convergence_fit_runs(self):
        """iterations=-1 triggers convergence_fit; just check it completes."""
        r = ResidualDecompositionSymmetric()
        r.fit(Xr, yr, model_class='ridge', iterations=-1, n_jobs=1)
        self.assertEqual(r.get_composition().shape, (N, N))


# ── 4. Visualisation ─────────────────────────────────────────────────────────
class TestVisualization(unittest.TestCase):

    def setUp(self):
        self.rshap = _fit(Xr, yr)
        self.labels = np.array(['A'] * (N // 2) + ['B'] * (N // 2))

    def tearDown(self):
        plt.close('all')

    def test_draw_heatmap_returns_fig_and_ax(self):
        fig, ax = draw_heatmap(self.rshap, self.labels)
        self.assertIsNotNone(fig)
        self.assertIsNotNone(ax)

    def test_draw_heatmap_accepts_external_ax(self):
        fig0, ax0 = plt.subplots()
        fig, ax = draw_heatmap(self.rshap, self.labels, ax=ax0)
        self.assertIs(ax, ax0)

    def test_cc_plot_no_coloring(self):
        plt.figure()
        sc = self.rshap.cc_plot()
        self.assertIsNotNone(sc)

    def test_cc_plot_continuous_coloring(self):
        plt.figure()
        sc = self.rshap.cc_plot(coloring=yr)
        self.assertIsNotNone(sc)

    def test_cc_plot_categorical_coloring(self):
        plt.figure()
        sc = self.rshap.cc_plot(coloring=self.labels)
        self.assertIsNotNone(sc)

    def test_cc_plot_group_coloring(self):
        plt.figure()
        sc = self.rshap.cc_plot(categorical_colouring=self.labels)
        self.assertIsNotNone(sc)


# ── Run all suites ────────────────────────────────────────────────────────────
suite = unittest.TestSuite()
loader = unittest.TestLoader()
for tc in (TestResolveModel, TestComputeMarginalContributions,
           TestRSHAPAttributes, TestRSHAPOutputs, TestVisualization):
    suite.addTests(loader.loadTestsFromTestCase(tc))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'='*60}")
print(f"UNIT TESTS: {result.testsRun} run  |  "
      f"{len(result.failures)} failures  |  {len(result.errors)} errors")
print('='*60)


test_case_insensitive (__main__.TestResolveModel.test_case_insensitive) ... ok
test_error_message_lists_valid_names (__main__.TestResolveModel.test_error_message_lists_valid_names) ... ok
test_explicit_params_preserved (__main__.TestResolveModel.test_explicit_params_preserved) ... ok
test_invalid_name_raises_value_error (__main__.TestResolveModel.test_invalid_name_raises_value_error) ... ok
test_logistic_alias (__main__.TestResolveModel.test_logistic_alias) ... ok
test_mlp_classification (__main__.TestResolveModel.test_mlp_classification) ... ok
test_mlp_regression (__main__.TestResolveModel.test_mlp_regression) ... ok
test_none_params_defaults_to_empty_dict (__main__.TestResolveModel.test_none_params_defaults_to_empty_dict) ... ok
test_passthrough_class_object (__main__.TestResolveModel.test_passthrough_class_object) ... ok
test_registry_has_expected_keys (__main__.TestResolveModel.test_registry_has_expected_keys) ... ok
test_rf_classification (__main__.TestResolveModel.test_rf_classi

Subprocess crashed with error: deliberate crash
Check rshap_error.log for details.


100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 69.15it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 66.45it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 71.81it/s]
ok
100%|███████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 173.42it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 74.35it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 24.78it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 73.04it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 24.20it/s]
ok
100%|███████████

Using convergence fit
Marginal change at iteration 11 is 0.09124021157344525
Marginal change at iteration 21 is 0.06857778155389317
Marginal change at iteration 31 is 0.03489376015773313
Marginal change at iteration 41 is 0.03769221445379057


ok
test_more_iterations_changes_phi (__main__.TestRSHAPOutputs.test_more_iterations_changes_phi) ... 

Marginal change at iteration 51 is 0.022140574242059858


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 72.42it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 23.84it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 21.79it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 23.90it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 24.89it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 63.95it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 68.92it/s]
ok
100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 72.68it/s]
ok
100%|███████████


UNIT TESTS: 52 run  |  0 failures  |  0 errors


### Algorithm Test Suite

In [5]:

import sys, warnings, time
sys.path.insert(0, "/path/to/RSHAP")
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")

import numpy as np
from sklearn.datasets import make_regression, make_classification
from RSHAP import ResidualDecompositionSymmetric

# ── Config ────────────────────────────────────────────────────────────────────
N_SAMPLES  = 40      # small enough to be fast, large enough to be meaningful
N_FEATURES = 5
ITERATIONS = 6       # symmetric pairs → 12 permutations per run
N_JOBS     = 1       # sequential; set to -1 to use all cores

MODELS = ['ridge', 'svm', 'rf', 'mlp', 'xgb']

np.random.seed(42)
Xr, yr = make_regression(
    n_samples=N_SAMPLES, n_features=N_FEATURES, noise=5.0, random_state=42)
Xc, yc = make_classification(
    n_samples=N_SAMPLES, n_features=N_FEATURES, n_classes=2,
    n_informative=3, random_state=42)

yr = yr.astype(np.float64)
yc = yc.astype(int)

# ── Helpers ───────────────────────────────────────────────────────────────────
def run_rshap(X, y, model, regression):
    r = ResidualDecompositionSymmetric()
    r.fit(X, y, model_class=model, iterations=ITERATIONS,
          regression=regression, n_jobs=N_JOBS)
    return r

def check(rshap, label):
    N = rshap.X1.shape[0]
    comp   = rshap.get_composition()
    contrib = rshap.get_contribution()
    issues = []
    if comp.shape   != (N, N): issues.append(f"composition shape {comp.shape} ≠ ({N},{N})")
    if contrib.shape != (N, N): issues.append(f"contribution shape {contrib.shape} ≠ ({N},{N})")
    if np.any(np.isnan(comp)):   issues.append("NaN in composition")
    if np.any(np.isnan(contrib)): issues.append("NaN in contribution")
    if np.all(comp == 0):         issues.append("composition is all-zero")
    if np.all(contrib == 0):      issues.append("contribution is all-zero")
    ok = len(issues) == 0
    icon = "✓" if ok else "✗"
    print(f"  {icon}  {label}")
    for issue in issues:
        print(f"       ↳ FAILED: {issue}")
    return ok

# ── Suite ─────────────────────────────────────────────────────────────────────
print("=" * 62)
print("ALGORITHM TEST SUITE")
print(f"  n={N_SAMPLES}, features={N_FEATURES}, {ITERATIONS} iterations ({ITERATIONS*2} permutations)")
print("=" * 62)

results = {}

# Regression ──────────────────────────────────────────────────────────────────
print(f"\n▶ REGRESSION   (make_regression, continuous target)")
print("-" * 45)
for model in MODELS:
    key = f"{model}/regression"
    t0 = time.perf_counter()
    try:
        rshap = run_rshap(Xr, yr, model, regression=True)
        elapsed = time.perf_counter() - t0
        ok = check(rshap, f"{model:8s}  [{elapsed:.1f}s]")
    except Exception as e:
        elapsed = time.perf_counter() - t0
        print(f"  ✗  {model:8s}  ERROR: {e}")
        ok = False
    results[key] = ok

# Classification ──────────────────────────────────────────────────────────────
print(f"\n▶ CLASSIFICATION   (make_classification, binary labels)")
print("-" * 45)
for model in MODELS:
    key = f"{model}/classification"
    t0 = time.perf_counter()
    try:
        rshap = run_rshap(Xc, yc, model, regression=False)
        elapsed = time.perf_counter() - t0
        ok = check(rshap, f"{model:8s}  [{elapsed:.1f}s]")
    except Exception as e:
        elapsed = time.perf_counter() - t0
        print(f"  ✗  {model:8s}  ERROR: {e}")
        ok = False
    results[key] = ok

# String alias ────────────────────────────────────────────────────────────────
print(f"\n▶ STRING ALIAS")
print("-" * 45)
for alias, reg in [('logistic', False), ('ridge', True)]:
    key = f"alias:{alias}"
    try:
        rshap = run_rshap(Xc if not reg else Xr, yc if not reg else yr,
                          alias, regression=reg)
        ok = check(rshap, f"'{alias}' alias ({'regression' if reg else 'classification'})")
    except Exception as e:
        print(f"  ✗  '{alias}' alias  ERROR: {e}")
        ok = False
    results[key] = ok

# Custom model_params ─────────────────────────────────────────────────────────
print(f"\n▶ CUSTOM model_params")
print("-" * 45)
custom_cases = [
    ('ridge', {'alpha': 10.0},              True,  "Ridge(alpha=10)"),
    ('svm',   {'C': 0.5, 'kernel': 'rbf'},  True,  "SVR(C=0.5, rbf)"),
    ('rf',    {'n_estimators': 10, 'max_depth': 3}, False, "RF(10 trees, depth=3)"),
    ('xgb',   {'n_estimators': 20, 'max_depth': 2}, True, "XGB(20 trees, depth=2)"),
]
for model, params, reg, label in custom_cases:
    key = f"custom/{label}"
    try:
        r = ResidualDecompositionSymmetric()
        r.fit(Xr if reg else Xc, yr if reg else yc,
              model_class=model, model_params=params,
              iterations=ITERATIONS, regression=reg, n_jobs=N_JOBS)
        ok = check(r, label)
    except Exception as e:
        print(f"  ✗  {label}  ERROR: {e}")
        ok = False
    results[key] = ok

# Summary ─────────────────────────────────────────────────────────────────────
passed = sum(results.values())
total  = len(results)
print(f"\n{'='*62}")
print(f"ALGORITHM SUITE: {passed}/{total} passed")
if passed < total:
    print("Failed:")
    for k, v in results.items():
        if not v:
            print(f"  ✗  {k}")
print("="*62)


ALGORITHM TEST SUITE
  n=40, features=5, 6 iterations (12 permutations)

▶ REGRESSION   (make_regression, continuous target)
---------------------------------------------


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 34.30it/s]


  ✓  ridge     [0.2s]


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 39.97it/s]


  ✓  svm       [0.2s]


...

  ✓  SVR(C=0.5, rbf)


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:02<00:00,  2.30it/s]


  ✓  RF(10 trees, depth=3)


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:01<00:00,  3.67it/s]

  ✓  XGB(20 trees, depth=2)

ALGORITHM SUITE: 16/16 passed


### Benchmark Test Suite

In [7]:

import sys, warnings
sys.path.insert(0, "/path/to/RSHAP")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC
from xgboost import XGBClassifier
from RSHAP import ResidualDecompositionSymmetric, draw_heatmap

np.random.seed(42)
ITERATIONS = 10
N_JOBS     = 1   # set to -1 to use all cores (may hit Windows paging limits)

# ── IMPORTANT — workflow note ─────────────────────────────────────────────────
# Each benchmark below follows the same two-step pattern:
#
#   Step 1. Train and evaluate YOUR prediction model using model_params
#   Step 2. Run RSHAP using the SAME model class and the SAME model_params
#
# RSHAP does not receive the trained model — it uses model_class and
# model_params to build its own fresh instances internally.  The params
# MUST match so that RSHAP's internal model behaves the same way as the
# model you are analysing.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 68)
print("BENCHMARK TEST SUITE")
print("Scientific datasets  |  one RSHAP capability demonstrated each")
print("=" * 68)

# ── Helper: robust OpenML loader ─────────────────────────────────────────────
def load_openml(data_id=None, name=None, version=1, target_col=-1):
    """Fetch an OpenML dataset; handle cases where ds.target is None."""
    if data_id is not None:
        ds = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
    else:
        ds = fetch_openml(name=name, version=version, as_frame=True, parser='auto')

    if ds.target is not None:
        X = ds.data.values.astype(np.float64)
        y = ds.target.values
        feat_names = ds.data.columns.tolist()
    else:
        frame = ds.frame
        col_idx = target_col if target_col >= 0 else frame.shape[1] + target_col
        feat_names = [c for i, c in enumerate(frame.columns) if i != col_idx]
        X = frame.iloc[:, [i for i in range(frame.shape[1]) if i != col_idx]].values.astype(np.float64)
        y = frame.iloc[:, col_idx].values

    return X, y, feat_names


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 1 — Concrete Compressive Strength  (regression, Ridge)
#
# CAPABILITY DEMONSTRATED: Influential instance identification
#   RSHAP's composition score (column sum of phi) measures how much each
#   training instance moves the residuals of every other instance.  High
#   positive scorers inflate errors; high negative scorers suppress them.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*68)
print("BENCHMARK 1 — Concrete Compressive Strength")
print("Capability: influential instance identification (composition scores)")
print("─"*68)

X1, y1_raw, feat_names1 = load_openml(data_id=4353, target_col=-1)
y1 = y1_raw.astype(np.float64)
print(f"\n  Dataset: {X1.shape[0]} mixes, {X1.shape[1]} ingredients, "
      f"target = compressive strength (MPa)")

# ── Step 1: define params, train and evaluate the prediction model ────────────
model_params1 = {'alpha': 1.0}
model1 = Ridge(**model_params1)
cv_r2 = cross_val_score(model1, X1, y1, cv=5, scoring='r2')
print(f"\n  Prediction model : Ridge(alpha={model_params1['alpha']})")
print(f"  Cross-val R²     : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

# ── Step 2: RSHAP — same model class, same model_params ──────────────────────
print(f"\n  Running RSHAP with Ridge, model_params={model_params1} ...")
rshap1 = ResidualDecompositionSymmetric()
rshap1.fit(X1, y1,
           model_class=Ridge,        # same class as prediction model
           model_params=model_params1,  # same params as prediction model
           iterations=ITERATIONS, regression=True, n_jobs=N_JOBS)

comp_scores = np.sum(rshap1.get_composition(), axis=0)
order       = np.argsort(comp_scores)
top5, bot5  = order[-5:][::-1], order[:5]

print(f"  Composition score range: [{comp_scores.min():.3f}, {comp_scores.max():.3f}]")

df_feat = pd.DataFrame(X1, columns=feat_names1)
df_feat['strength_MPa'] = y1
df_feat['comp_score']   = comp_scores
show_cols = feat_names1[:4] + ['strength_MPa', 'comp_score']

print("\n  Top-5 most positively influential mixes (raise residuals of others):")
print(df_feat.iloc[top5][show_cols].to_string(index=True))
print("\n  Top-5 most negatively influential mixes (suppress residuals of others):")
print(df_feat.iloc[bot5][show_cols].to_string(index=True))

score_corrs = [abs(np.corrcoef(X1[:, j], comp_scores)[0, 1]) for j in range(X1.shape[1])]
top_feat    = feat_names1[int(np.argmax(score_corrs))]
print(f"\n  Feature most correlated with influence score: '{top_feat}' "
      f"(|r| = {max(score_corrs):.3f})")
print("\n  ✓ PASS" if comp_scores.std() > 0 else "\n  ✗ FAIL — all scores identical")


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 2 — NASA Airfoil Self-Noise  (regression, Random Forest)
#
# CAPABILITY DEMONSTRATED: CC plot with continuous target coloring
#   Aerodynamically extreme aerofoils — highest and lowest dB readings —
#   should appear as outliers in RSHAP space, away from the central cloud.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*68)
print("BENCHMARK 2 — NASA Airfoil Self-Noise")
print("Capability: CC plot — locating outliers by sound pressure level")
print("─"*68)

_url2 = ("https://archive.ics.uci.edu/ml/machine-learning-databases"
         "/00291/airfoil_self_noise.dat")
_cols2 = ['frequency', 'angle', 'chord_length', 'velocity', 'displacement', 'dB']
try:
    df2 = pd.read_csv(_url2, sep='\t', header=None, names=_cols2)
except Exception as _e:
    raise RuntimeError(f"Could not download Airfoil dataset: {_e}\n"
                       "Check your internet connection and try again.")
X2 = df2.iloc[:, :-1].values.astype(np.float64)
y2 = df2['dB'].values.astype(np.float64)
print(f"\n  Dataset: {X2.shape[0]} aerofoil conditions, {X2.shape[1]} aerodynamic features")

# ── Step 1: define params, train and evaluate the prediction model ────────────
model_params2 = {'n_estimators': 100, 'n_jobs': 1}
model2 = RandomForestRegressor(**model_params2)
cv_r2_2 = cross_val_score(model2, X2, y2, cv=3, scoring='r2')
print(f"\n  Prediction model : RandomForestRegressor(n_estimators={model_params2['n_estimators']})")
print(f"  Cross-val R²     : {cv_r2_2.mean():.3f} ± {cv_r2_2.std():.3f}")

# ── Step 2: RSHAP — same model class, same model_params ──────────────────────
print(f"\n  Running RSHAP with RandomForestRegressor, model_params={model_params2} ...")
rshap2 = ResidualDecompositionSymmetric()
rshap2.fit(X2, y2,
           model_class=RandomForestRegressor,  # same class as prediction model
           model_params=model_params2,          # same params as prediction model
           iterations=ITERATIONS, regression=True, n_jobs=N_JOBS)

x_cc2 = np.sum(rshap2.get_composition(), axis=0)
y_cc2 = np.sum(rshap2.get_contribution(), axis=1)

fig2, ax2 = plt.subplots(figsize=(6, 5))
sc = ax2.scatter(x_cc2, y_cc2, c=y2, cmap='viridis', alpha=0.6, s=15)
cb = fig2.colorbar(sc, ax=ax2)
cb.set_label('Sound pressure level (dB)', fontsize=11)
ax2.axhline(0, color='orange', lw=0.8); ax2.axvline(0, color='red', lw=0.8)
ax2.set_xlabel('Composition sum', fontsize=12)
ax2.set_ylabel('Contribution sum', fontsize=12)
ax2.set_title('Airfoil Self-Noise — CC plot\n(color = measured dB)', fontsize=12)
fig2.tight_layout()
fig2.savefig('benchmark2_airfoil_ccplot.png', dpi=120)
plt.show()

dB_extreme  = (y2 > np.percentile(y2, 90)) | (y2 < np.percentile(y2, 10))
mag_extreme = np.sqrt(x_cc2[dB_extreme]**2  + y_cc2[dB_extreme]**2).mean()
mag_central = np.sqrt(x_cc2[~dB_extreme]**2 + y_cc2[~dB_extreme]**2).mean()
print(f"  Mean RSHAP distance from origin — extreme dB: {mag_extreme:.4f}")
print(f"  Mean RSHAP distance from origin — central dB: {mag_central:.4f}")
print(f"  Ratio (extreme / central): {mag_extreme / mag_central:.2f}x")
print(f"  Plot saved → benchmark2_airfoil_ccplot.png")
print("\n  ✓ PASS" if mag_extreme > mag_central else
      "\n  NOTE — extreme dB instances not more displaced (try more iterations)")


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 3 — QSAR Biodegradation  (classification, SVM)
#
# CAPABILITY DEMONSTRATED: Class separation in CC space
#   CC plot with categorical class coloring shows whether biodegradable and
#   non-biodegradable molecules occupy distinct regions of RSHAP space.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*68)
print("BENCHMARK 3 — QSAR Biodegradation")
print("Capability: CC plot class separation (biodegradable vs. not)")
print("─"*68)

X3, y3_raw, _ = load_openml(data_id=1494)
le3  = LabelEncoder()
y3   = le3.fit_transform(y3_raw)
class_names = le3.classes_
print(f"\n  Dataset: {X3.shape[0]} molecules, {X3.shape[1]} molecular descriptors")

# ── Step 1: define params, train and evaluate the prediction model ────────────
model_params3 = {'C': 1.0}
model3 = SVC(**model_params3)
cv_acc3 = cross_val_score(model3, X3, y3, cv=3, scoring='accuracy')
print(f"\n  Prediction model : SVC(C={model_params3['C']})")
print(f"  Cross-val accuracy: {cv_acc3.mean():.3f} ± {cv_acc3.std():.3f}")

# ── Step 2: RSHAP — same model class, same model_params ──────────────────────
print(f"\n  Running RSHAP with SVC, model_params={model_params3} ...")
rshap3 = ResidualDecompositionSymmetric()
rshap3.fit(X3, y3,
           model_class=SVC,          # same class as prediction model
           model_params=model_params3,  # same params as prediction model
           iterations=ITERATIONS, regression=False, n_jobs=N_JOBS)

x_cc3 = np.sum(rshap3.get_composition(), axis=0)
y_cc3 = np.sum(rshap3.get_contribution(), axis=1)

fig3, ax3 = plt.subplots(figsize=(6, 5))
viridis = plt.get_cmap('viridis')
colors  = {0: viridis(0.15), 1: viridis(0.85)}
for cls_idx, cls_name in enumerate(class_names):
    mask = y3 == cls_idx
    ax3.scatter(x_cc3[mask], y_cc3[mask], c=[colors[cls_idx]],
                label=cls_name, alpha=0.6, s=18)
ax3.axhline(0, color='orange', lw=0.8); ax3.axvline(0, color='red', lw=0.8)
ax3.set_xlabel('Composition sum', fontsize=12)
ax3.set_ylabel('Contribution sum', fontsize=12)
ax3.set_title('QSAR Biodegradation — CC plot\n(color = class)', fontsize=12)
ax3.legend(title='Class', fontsize=10)
fig3.tight_layout()
fig3.savefig('benchmark3_biodeg_ccplot.png', dpi=120)
plt.show()

for cls_idx, cls_name in enumerate(class_names):
    mask = y3 == cls_idx
    cx, cy = x_cc3[mask].mean(), y_cc3[mask].mean()
    print(f"  Class '{cls_name}':  centroid = ({cx:.4f}, {cy:.4f})")
centroid_sep = np.sqrt(
    (x_cc3[y3==0].mean() - x_cc3[y3==1].mean())**2 +
    (y_cc3[y3==0].mean() - y_cc3[y3==1].mean())**2)
print(f"\n  Centroid separation in CC space: {centroid_sep:.4f}")
print(f"  Plot saved → benchmark3_biodeg_ccplot.png")
print("\n  ✓ PASS" if centroid_sep > 0 else "\n  ✗ FAIL — centroids coincide")


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 4 — Wine Quality Red  (classification, XGBoost)
#
# CAPABILITY DEMONSTRATED: Inter-class contribution heatmap
#   draw_heatmap() shows how strongly instances of one quality tier influence
#   the residuals of another.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*68)
print("BENCHMARK 4 — Wine Quality Red")
print("Capability: inter-class contribution heatmap (high vs. low quality)")
print("─"*68)

X4, y4_raw, _ = load_openml(data_id=40691)
y4 = (y4_raw.astype(float) >= 6).astype(int)
print(f"\n  Dataset: {X4.shape[0]} red wine samples, {X4.shape[1]} physicochemical features")

# ── Step 1: define params, train and evaluate the prediction model ────────────
model_params4 = {'n_estimators': 100, 'max_depth': 4, 'verbosity': 0}
model4 = XGBClassifier(**model_params4)
cv_acc4 = cross_val_score(model4, X4, y4, cv=3, scoring='accuracy')
print(f"\n  Prediction model : XGBClassifier(n_estimators={model_params4['n_estimators']}, "
      f"max_depth={model_params4['max_depth']})")
print(f"  Cross-val accuracy: {cv_acc4.mean():.3f} ± {cv_acc4.std():.3f}")

# ── Step 2: RSHAP — same model class, same model_params ──────────────────────
print(f"\n  Running RSHAP with XGBClassifier, model_params={model_params4} ...")
rshap4 = ResidualDecompositionSymmetric()
rshap4.fit(X4, y4,
           model_class=XGBClassifier,  # same class as prediction model
           model_params=model_params4,  # same params as prediction model
           iterations=ITERATIONS, regression=False, n_jobs=N_JOBS)

quality_labels = np.where(y4 == 1, 'high (>=6)', 'low (<6)')
fig4, ax4 = plt.subplots(figsize=(5, 4))
draw_heatmap(rshap4, quality_labels, ax=ax4, fontsizes=13)
ax4.set_title('Wine Quality — inter-class contribution\n(high vs. low quality)', fontsize=12)
fig4.tight_layout()
fig4.savefig('benchmark4_wine_heatmap.png', dpi=120)
plt.show()

comp4    = rshap4.get_contribution()
hi_mask  = quality_labels == 'high (>=6)'
lo_mask  = quality_labels == 'low (<6)'
hi_to_lo = comp4[np.ix_(hi_mask, lo_mask)].mean()
lo_to_hi = comp4[np.ix_(lo_mask, hi_mask)].mean()
print(f"  High-quality (>=6): {hi_mask.sum()}  |  Low-quality (<6): {lo_mask.sum()}")
print(f"  Mean contribution — high-quality → low-quality: {hi_to_lo:.5f}")
print(f"  Mean contribution — low-quality  → high-quality: {lo_to_hi:.5f}")
print(f"  Asymmetry ratio |hi→lo| / |lo→hi|: "
      f"{abs(hi_to_lo)/max(abs(lo_to_hi), 1e-12):.2f}x")
print(f"  Plot saved → benchmark4_wine_heatmap.png")
print("\n  ✓ PASS" if hi_to_lo != lo_to_hi else
      "\n  NOTE — inter-class contributions symmetric (try more iterations)")


# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "="*68)
print("BENCHMARK SUMMARY")
print("-"*68)
rows = [
    ("Concrete Compressive Strength", "Ridge",          "regression",     "Influential instance identification"),
    ("NASA Airfoil Self-Noise",        "RandomForest",   "regression",     "CC plot — outlier detection by dB level"),
    ("QSAR Biodegradation",            "SVC",            "classification", "CC plot — class separation in RSHAP space"),
    ("Wine Quality Red",               "XGBClassifier",  "classification", "Inter-class contribution heatmap"),
]
for name, model, task, demo in rows:
    print(f"  ✓  {name:<34s} ({model:15s}, {task:<14s}) → {demo}")
print("="*68)


BENCHMARK TEST SUITE
Scientific datasets  |  one RSHAP capability demonstrated each

════════════════════════════════════════════════════════════════════
BENCHMARK 1 — Concrete Compressive Strength
Capability: influential instance identification (composition scores)
────────────────────────────────────────────────────────────────────

  Dataset: 1030 mixes, 8 ingredients, target = compressive strength (MPa)

  Prediction model : Ridge(alpha=1.0)
  Cross-val R²     : 0.461 ± 0.093

  Running RSHAP with Ridge, model_params={'alpha': 1.0} ...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.24it/s]


  Composition score range: [-34.452, 28.653]

  Top-5 most positively influential mixes (raise residuals of others):
     Cement (component 1)(kg in a m^3 mixture)  Blast Furnace Slag (component 2)(kg in a m^3 mixture)  Fly Ash (component 3)(kg in a m^3 mixture)  Water  (component 4)(kg in a m^3 mixture)  strength_MPa  comp_score
114                                      362.6                                                  189.0                                         0.0                                      164.9         22.90   28.652964
56                                       475.0                                                    0.0                                         0.0                                      228.0         41.93   28.007200
610                                      236.0                                                    0.0                                         0.0                                      193.0         25.08   27.994849
477                    

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [54:47<00:00, 328.71s/it]


  Mean RSHAP distance from origin — extreme dB: 66.8566
  Mean RSHAP distance from origin — central dB: 35.2677
  Ratio (extreme / central): 1.90x
  Plot saved → benchmark2_airfoil_ccplot.png

  ✓ PASS

════════════════════════════════════════════════════════════════════
BENCHMARK 3 — QSAR Biodegradation
Capability: CC plot class separation (biodegradable vs. not)
────────────────────────────────────────────────────────────────────

  Dataset: 1055 molecules, 41 molecular descriptors

  Prediction model : SVC(C=1.0)
  Cross-val accuracy: 0.772 ± 0.034

  Running RSHAP with SVC, model_params={'C': 1.0} ...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [06:20<00:00, 38.05s/it]


  Class '1':  centroid = (0.0848, 0.0983)
  Class '2':  centroid = (0.3388, 0.0056)

  Centroid separation in CC space: 0.2703
  Plot saved → benchmark3_biodeg_ccplot.png

  ✓ PASS

════════════════════════════════════════════════════════════════════
BENCHMARK 4 — Wine Quality Red
Capability: inter-class contribution heatmap (high vs. low quality)
────────────────────────────────────────────────────────────────────

  Dataset: 1599 red wine samples, 11 physicochemical features

  Prediction model : XGBClassifier(n_estimators=100, max_depth=4)
  Cross-val accuracy: 0.709 ± 0.009

  Running RSHAP with XGBClassifier, model_params={'n_estimators': 100, 'max_depth': 4, 'verbosity': 0} ...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [07:31<00:00, 45.18s/it]


  High-quality (>=6): 855  |  Low-quality (<6): 744
  Mean contribution — high-quality → low-quality: 0.00040
  Mean contribution — low-quality  → high-quality: -0.00046
  Asymmetry ratio |hi→lo| / |lo→hi|: 0.87x
  Plot saved → benchmark4_wine_heatmap.png

  ✓ PASS

BENCHMARK SUMMARY
--------------------------------------------------------------------
  ✓  Concrete Compressive Strength      (Ridge          , regression    ) → Influential instance identification
  ✓  NASA Airfoil Self-Noise            (RandomForest   , regression    ) → CC plot — outlier detection by dB level
  ✓  QSAR Biodegradation                (SVC            , classification) → CC plot — class separation in RSHAP space
  ✓  Wine Quality Red                   (XGBClassifier  , classification) → Inter-class contribution heatmap
